# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulm111/ML-Assignement01/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Abdulm111/ML-Assignement01"
REPO_DIR = "ML-Assignement01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/ML-Assignement01/ML-Assignement01
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

**Provisional lane: CTR / Engagement Opportunity Scoring.**

The starter dataset shows a clean, monotonic drop in mean CTR as position tier gets worse
(page_1 = 0.355% down to deep = 0.055%), which means CTR only makes sense compared *within*
the same tier  a page at position 15 should never be judged against a page at position 2.
There's also real volume to work with: 12,023 of the 30,000 starter pages sit at both
meaningful exposure (impressions_90d >= 500) and a top-20 position, so there's a large enough
pool to rank fairly. I'm choosing this over Lane 2 (Refresh Scoring) because CTR-by-tier gives
a more objective "expected vs actual" comparison to build a score from, rather than a mix of
loosely related rules. See the numbers below.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[df["impressions_90d"] >= 100]
ctr_by_tier = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Mean CTR by position tier (impressions_90d >= 100):")
print(ctr_by_tier.round(3))

pool = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)]
print(f"\nReview pool (impressions_90d >= 500, position 1-20): {len(pool)} of {len(df)} pages")

Mean CTR by position tier (impressions_90d >= 100):
position_tier
page_1      0.355
top_3       0.334
striking    0.256
page_3_5    0.142
deep        0.055
Name: ctr, dtype: float64

Review pool (impressions_90d >= 500, position 1-20): 12023 of 30000 pages


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

**Decision:** Which visible pages are under-capturing clicks relative to other pages at the
same position tier, and should be reviewed first.

**Who acts, and how:** A content editor works down a ranked review queue, checking title, meta
description, and snippet against query intent for each flagged page.

**Cost of a wrong call:** Wasted editor time on a page whose "low" CTR is really just low-volume
noise — a few unlucky clicks on 100 impressions can look identical to a real pattern. A missed
flag has a cost too: a genuinely underperforming page goes another review cycle without a look,
even if a review might have surfaced something worth fixing.

**Why data/ML, not just a rule:** A flat rule like "CTR < 0.5%" is too blunt to work as a
priority list — see below. A tier-relative score is a natural next step to test; whether it
narrows the list meaningfully, and whether other signals (content type, age, intent) predict the
gap better than tier alone, is exactly what the next few weeks are for finding out.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Test the flat rule the starter baseline already uses, against the review pool above
flagged = pool[pool["ctr"] < 0.5]
print(f"Flagged by flat CTR < 0.5 rule: {len(flagged)} of {len(pool)} pages "
      f"({len(flagged)/len(pool)*100:.1f}% of the pool)")

Flagged by flat CTR < 0.5 rule: 9759 of 12023 pages (81.2% of the pool)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

print(f"\nRows with impressions_90d < 100 (excluded as too noisy for tier comparison): "
      f"{(df['impressions_90d']<100).sum()} of {len(df)}")

print("\nPages per position tier:")
print(df["position_tier"].value_counts())

Dataset: 30000 rows, 44 columns

Rows with impressions_90d < 100 (excluded as too noisy for tier comparison): 7994 of 30000

Pages per position tier:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

This project will report **observed, directional** patterns only — for example, "pages sitting
below their tier's average CTR" — never "pages Google is punishing" or anything about a ranking
algorithm. It is **decision-support**, not a guarantee: a low relative CTR means a page is worth
a look, not that fixing it will definitely recover clicks. I will not claim a title/meta rewrite
*caused* any CTR change unless I run an explicit before/after comparison. All numbers stay
aggregated and pseudonymized — no client names, raw URLs, or raw queries anywhere in the
notebook or any published output.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unsafe_keywords = ["url", "query", "title", "domain", "keyword"]
touched_columns = ["position_tier", "ctr", "impressions_90d", "avg_position", "client_id"]
unsafe_used = [c for c in touched_columns if any(k in c.lower() for k in unsafe_keywords)]
print("Unsafe fields touched in this notebook:", unsafe_used if unsafe_used else "none")

Unsafe fields touched in this notebook: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.